In [10]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [11]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [12]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages,"```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [13]:
dateset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dateset, f, indent=2)

In [14]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [15]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [16]:
def run_test_case(test_case):
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output, 
        "test_case": test_case, 
        "score": score,
        "reasoning": reasoning
    }

In [17]:
from statistics import mean

def run_eval(dataset):
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [18]:

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 7.0


In [19]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Region Extraction Function\n\nHere's a Python function that extracts the AWS region from an S3 bucket URL:\n\n```python\nimport re\n\ndef extract_aws_region_from_s3_url(url: str) -> str:\n    \"\"\"\n    Extract the AWS region from an S3 bucket URL.\n    \n    Supports multiple S3 URL formats:\n    - Virtual-hosted style: s3://my-bucket.s3.us-west-2.amazonaws.com/key\n    - Path style: s3://s3.us-west-2.amazonaws.com/my-bucket/key\n    - Legacy: s3://my-bucket.s3.amazonaws.com/key (returns 'us-east-1')\n    \n    Args:\n        url (str): The S3 bucket URL\n        \n    Returns:\n        str: The AWS region code (e.g., 'us-west-2')\n        \n    Raises:\n        ValueError: If region cannot be extracted from URL\n    \"\"\"\n    \n    # Pattern for virtual-hosted style: bucket.s3.region.amazonaws.com\n    virtual_hosted_pattern = r'\\.s3\\.([a-z0-9\\-]+)\\.amazonaws\\.com'\n    match = re.search(virtual_hosted_pattern, url)\n    if match:\n        return